## COS711 Assignment 3
Automatically labelling radio sources with deep learning

University of Pretoria, Computer Science Department
COS711: Artificial Intelligence II (AI)

By: Lerato Letsepe (u25468023), Aidan Govender (u22520485), Ettienne Van Zyl (u19012366)

**Project Overview**: This notebook presents a comprehensive deep learning pipeline for the automatic classification of radio sources from the MeerKAT Galaxy Cluster Legacy Survey (MGCLS). The primary challenges include a small labeled dataset, significant class imbalance, and the multi-label nature of the classification task.

Our approach is structured as follows:


1. Data Preparation: We meticulously clean the labels, handle the "Should be Discarded" class as a valid category , and use an efficient cKDTree algorithm to match astronomical coordinates to image files.

2. Baseline Model: We train a powerful pre-trained model (EfficientNetV2) on the initial labeled data, establishing a performance baseline. We explicitly address the class imbalance using a weighted loss function.

3. Enhancement via Pseudo-Labelling: To leverage the large set of unlabeled data, we implement a pseudo-labelling pipeline. We use our baseline model to predict labels for unlabeled images, select the most confident predictions, and then fine-tune a new model on this augmented dataset.


4. Evaluation & Comparison: We rigorously compare the performance of the baseline model against the enhanced pseudo-labeled model to demonstrate the effectiveness of our approach.

5. Final Deliverables: We generate the required test_labels.csv and generated_labels.csv files using our best-performing model.

In [1]:
# Uncomment if you are running before unzipping the images folders. This will extract in same BASE directory
# import os, zipfile
# from pathlib import Path

# BASE = Path("/content/drive/MyDrive/UP Honours/COS711 - AI/Assignment 3/Data")
# BASE.mkdir(parents=True, exist_ok=True)

# # Unzip if present
# for z in ["typ.zip", "exo.zip", "unl.zip"]:
#     zpath = BASE / z
#     if zpath.exists():
#         with zipfile.ZipFile(zpath, 'r') as zf:
#             zf.extractall(BASE / z.replace('.zip',''))
#     else:
#         print("Warning: missing", zpath)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
## Uncomment if running in memory contrained env
# import os
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [4]:
# Unomment out if you want to do the best parameter searching
# !pip install optuna -q
# import optuna

In [5]:
# Cell 1: Project Setup & Imports

# Basic imports for data handling, numerics, and plotting.
import os, zipfile
import sys
import random
import math
import re
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from fastprogress import progress_bar
from PIL import Image, UnidentifiedImageError
from scipy.spatial import cKDTree

# Scikit-learn for data splitting and multi-label encoding
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import roc_auc_score, multilabel_confusion_matrix, classification_report, f1_score, precision_score, recall_score

# PyTorch for deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# Mixed-precision training for performance
try:
    from torch.cuda.amp import autocast, GradScaler
except ImportError:
    print("Warning: torch.cuda.amp not found. Running without mixed precision.")
    # Dummy classes if amp is not available
    class autocast:
        def __enter__(self): pass
        def __exit__(self, exc_type, exc_val, exc_tb): pass
    class GradScaler:
        def scale(self, loss): return loss
        def step(self, optimizer): optimizer.step()
        def update(self): pass

# Deterministic Setup for Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch Version: 2.8.0+cu126
CUDA Available: True


In [6]:
# Cell 1.5 - Image processing related imports
from pathlib import Path
import cv2
import json
from typing import List, Dict, Any, Optional, Tuple, Iterable
from dataclasses import dataclass
import numpy as np
from PIL import Image



# Utility to list images
def list_images(root: Path, patterns: List[str]) -> List[Path]:
    files: List[Path] = []
    for pat in patterns:
        files.extend(sorted(root.rglob(pat)))
    seen = set()
    unique = []
    for f in files:
        if f not in seen:
            unique.append(f)
            seen.add(f)
    return unique

In [7]:
from abc import ABC, abstractmethod

class IImagePreprocessor(ABC):
    @abstractmethod
    def preprocess(self, img_bgr: np.ndarray) -> Dict[str, Any]:
        """Return dict with keys:
            'processed' -> processed grayscale uint8 image
            'mask' -> binary uint8 mask (optional)
            'bbox' -> (x, y, w, h) used for crop
            'metrics' -> quality metrics dict
        """
        raise NotImplementedError

class IImageRepository(ABC):
    @abstractmethod
    def load_image(self, path: Path) -> 'np.ndarray':
        pass
    @abstractmethod
    def save_image(self, path: Path, image: 'np.ndarray') -> None:
        pass
    @abstractmethod
    def save_json(self, path: Path, data: dict) -> None:
        pass

In [8]:
@dataclass(frozen=True)
class ImageSample:
    path: Path
    id: str

@dataclass
class ProcessingConfig:
    log_transform: bool = True
    log_c: float = 0.7
    contrast_stretch: bool = True

    denoise_method: str = "median"  # "gaussian" | "median" | "none"
    denoise_sigma: float = 0.0
    denoise_ksize: int = 3  # must be odd

    background_method: str = "tophat"  # "tophat" | "none"
    background_se_radius: int = 12

    sharpen_method: str = "unsharp"  # "unsharp" | "none"
    unsharp_amount: float = 0.22
    unsharp_radius: int = 1  # kernel size for blur used in unsharp

    mask_threshold: str = "kapur"  # "otsu" | "kapur" | "none"
    closing_radius: int = 2

    crop_margin: int = 8  # pixels
    resize_to: int = None  # final square size (pixels)
    interpolation: str = "bicubic"  # "bilinear" | "bicubic" | "nearest"

    save_mask: bool = False
    quality_gate: bool = True
    min_edge_density: float = 0.02  # edges / pixel
    min_laplacian_var: float = 20.0

    def to_dict(self) -> Dict[str, Any]:
        return self.__dict__

In [9]:

def preprocess_dataset(
    input_dir: Path,
    output_subdir: str,
    repo: "IImageRepository",
    preprocessor: "IImagePreprocessor",
    config: "ProcessingConfig",
    patterns: List[str] = None
) -> None:
    patterns = patterns or ["*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"]
    samples: List["ImageSample"] = [ImageSample(p, p.stem) for p in list_images(input_dir, patterns)]
    out_dir = input_dir / output_subdir
    masks_dir = out_dir / "masks"
    meta_dir = out_dir / "meta"
    out_dir.mkdir(parents=True, exist_ok=True)
    if config.save_mask:
        masks_dir.mkdir(parents=True, exist_ok=True)
    meta_dir.mkdir(parents=True, exist_ok=True)

    total = len(samples)
    print(f"Found {total} images under {input_dir}")

    ok = err = 0
    pbar = progress_bar(samples, total=total)
    for s in pbar:
        try:
            img = repo.load_image(s.path)
            result = preprocessor.preprocess(img)
            processed = result["processed"]
            repo.save_image(out_dir / f"{s.id}.png", processed)

            if config.save_mask and result.get("mask") is not None:
                repo.save_image(masks_dir / f"{s.id}_mask.png", result["mask"])

            repo.save_json(meta_dir / f"{s.id}.json", {
                "source": str(s.path),
                "bbox": result.get("bbox"),
                "metrics": result.get("metrics"),
                "config": config.to_dict()
            })
            ok += 1
        except Exception as e:
            err += 1
            # prints render below the bar in notebooks
            print(f"ERROR processing {s.path}: {e}")
        finally:
            # show rolling counts to the right of the bar
            pbar.comment = f"ok: {ok} | err: {err}"

    print(f"Done. Success: {ok}, Errors: {err}. Output: {out_dir}")


class FileSystemImageRepository(IImageRepository):
    """Simple implementation that loads/saves images and JSON metadata to the local filesystem.
    - load_image returns a BGR uint8 numpy array (compatible with OpenCV pipelines)
    - save_image accepts numpy arrays (grayscale or BGR) and writes PNG files
    - save_json writes JSON metadata
    """
    def load_image(self, path: Path) -> np.ndarray:
        p = Path(path)
        if not p.exists():
            raise FileNotFoundError(p)
        # Use OpenCV (BGR) if available, otherwise PIL fallback (converted to BGR)
        try:
            arr = cv2.imdecode(np.fromfile(str(p), dtype=np.uint8), cv2.IMREAD_COLOR)
            if arr is None:
                raise ValueError("cv2 failed to read image")
            return arr
        except Exception:
            img = Image.open(p).convert('RGB')
            arr = np.array(img)[:, :, ::-1].copy()  # RGB->BGR
            return arr

    def save_image(self, path: Path, image: np.ndarray) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        out = image
        if isinstance(out, np.ndarray) and out.dtype != np.uint8:
            out = np.clip(out, 0, 255).astype(np.uint8)
        if out.ndim == 2:
            try:
                cv2.imencode('.png', out)[1].tofile(str(path))
                return
            except Exception:
                Image.fromarray(out).save(str(path))
                return
        if out.ndim == 3:
            try:
                cv2.imencode('.png', out)[1].tofile(str(path))
                return
            except Exception:
                Image.fromarray(out[:, :, ::-1]).save(str(path))  # BGR->RGB for PIL
                return
        Image.fromarray(np.uint8(out)).save(str(path))

    def save_json(self, path: Path, data: dict) -> None:
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w', encoding='utf-8') as fh:
            json.dump(data, fh, indent=2)


class OpenCVPreprocessor(IImagePreprocessor):
    """Color-preserving preprocessor.
    - Works in BGR throughout.
    - Uses luminance (Y) only for masking/metrics to avoid color shifts.
    Returns: {'processed'(BGR uint8), 'mask'(binary uint8 or None), 'bbox', 'metrics'}
    """
    def __init__(self, config: "ProcessingConfig"):
        self.cfg = config

    def _ensure_odd(self, k: int) -> int:
        return k if k % 2 == 1 else max(1, k - 1)

    def _luminance(self, bgr: np.ndarray) -> np.ndarray:
        ycrcb = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)
        return ycrcb[:, :, 0]

    def preprocess(self, img_bgr: np.ndarray) -> Dict[str, Any]:
        cfg = self.cfg

        # ---- 1) Denoise on color ----
        proc = img_bgr.copy()
        if cfg.denoise_method == 'gaussian':
            k = self._ensure_odd(cfg.denoise_ksize)
            proc = cv2.GaussianBlur(proc, (k, k), cfg.denoise_sigma)
        elif cfg.denoise_method == 'median':
            k = self._ensure_odd(cfg.denoise_ksize)
            proc = cv2.medianBlur(proc, k)

        # ---- 2) Background removal (Top-hat) on luminance, then merge back ----
        if cfg.background_method == 'tophat':
            ksize = max(3, int(cfg.background_se_radius))
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ksize, ksize))
            ycrcb = cv2.cvtColor(proc, cv2.COLOR_BGR2YCrCb)
            Y = ycrcb[:, :, 0]
            tophat = cv2.morphologyEx(Y, cv2.MORPH_TOPHAT, kernel)
            Y_enh = cv2.add(Y, tophat)
            ycrcb[:, :, 0] = np.clip(Y_enh, 0, 255).astype(np.uint8)
            proc = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)

        # ---- 3) Optional log transform (per-channel, safe scaling, NumPy 2.0) ----
        if cfg.log_transform:
            f = proc.astype(np.float32)
            f = np.log1p(f * cfg.log_c)
            # Scale per-channel independently to avoid tint shifts
            for c in range(3):
                ch = f[:, :, c]
                rng = float(np.ptp(ch))  # NumPy 2.0-safe (replaces ch.ptp())
                if rng < 1e-12:          # avoid divide-by-zero / NaNs
                    f[:, :, c] = 0.0
                else:
                    ch_min = float(np.min(ch))
                    f[:, :, c] = 255.0 * (ch - ch_min) / (rng + 1e-9)
            proc = np.clip(f, 0, 255).astype(np.uint8)

        # ---- 4) Contrast enhancement (luminance equalization; linear per-channel fallback) ----
        if cfg.contrast_stretch:
            try:
                ycrcb = cv2.cvtColor(proc, cv2.COLOR_BGR2YCrCb)
                ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
                proc = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2BGR)
            except Exception:
                out = proc.astype(np.float32)
                for c in range(3):
                    p1, p99 = np.percentile(out[:, :, c], (1, 99))
                    denom = (p99 - p1) if (p99 - p1) > 1e-9 else 1.0
                    out[:, :, c] = np.clip((out[:, :, c] - p1) * 255.0 / denom, 0, 255)
                proc = out.astype(np.uint8)

        # ---- 5) Sharpen (unsharp mask) on color ----
        if cfg.sharpen_method == 'unsharp':
            r = self._ensure_odd(cfg.unsharp_radius * 2 + 1)
            blurred = cv2.GaussianBlur(proc, (r, r), 0)
            proc = cv2.addWeighted(proc, 1.0 + cfg.unsharp_amount, blurred, -cfg.unsharp_amount, 0)
            proc = np.clip(proc, 0, 255).astype(np.uint8)

        # ---- 6) Mask (from luminance of *processed* image) ----
        mask = None
        if cfg.mask_threshold in ('otsu', 'kapur'):
            try:
                lum = self._luminance(proc)
                # (If 'kapur' is configured, still using Otsu here as a robust default)
                _, th = cv2.threshold(lum, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                mask = th
            except Exception:
                mask = None

        # Morphological closing
        if mask is not None and cfg.closing_radius > 0:
            kr = max(1, int(cfg.closing_radius))
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kr, kr))
            mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

        # ---- 7) BBox from mask (fallback: full image) ----
        h, w = proc.shape[:2]
        bbox = None
        if mask is not None and mask.sum() > 0:
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                c = max(contours, key=cv2.contourArea)
                x, y, bw, bh = cv2.boundingRect(c)
                x = max(0, x - cfg.crop_margin)
                y = max(0, y - cfg.crop_margin)
                bw = min(w - x, bw + 2 * cfg.crop_margin)
                bh = min(h - y, bh + 2 * cfg.crop_margin)
                bbox = (int(x), int(y), int(bw), int(bh))
        if bbox is None:
            bbox = (0, 0, w, h)

        # ---- 8) Crop & resize (color) ----
        x, y, bw, bh = bbox
        cropped = proc[y:y+bh, x:x+bw]

        target = max(1, int(cfg.resize_to))
        interp_map = {'bicubic': cv2.INTER_CUBIC, 'bilinear': cv2.INTER_LINEAR, 'nearest': cv2.INTER_NEAREST}
        interp = interp_map.get(cfg.interpolation, cv2.INTER_CUBIC)
        processed = cv2.resize(cropped, (target, target), interpolation=interp)

        # ---- 9) Metrics (on luminance of processed) ----
        lum_proc = self._luminance(processed)
        edges = cv2.Canny(lum_proc, 50, 150)
        edge_density = float(edges.sum() / 255) / (lum_proc.size)
        lap_var = float(cv2.Laplacian(lum_proc, cv2.CV_64F).var())
        metrics = {
            'edge_density': edge_density,
            'laplacian_var': lap_var,
            'orig_shape': (h, w, 3),
            'cropped_shape': processed.shape
        }

        metrics['quality_pass'] = True
        if cfg.quality_gate:
            if cfg.min_edge_density is not None and edge_density < cfg.min_edge_density:
                metrics['quality_pass'] = False
            if cfg.min_laplacian_var is not None and lap_var < cfg.min_laplacian_var:
                metrics['quality_pass'] = False

        # ---- 10) Resize mask to match output (if present) ----
        out_mask = None
        if mask is not None:
            mask_cropped = mask[y:y+bh, x:x+bw]
            out_mask = cv2.resize(mask_cropped, (target, target), interpolation=cv2.INTER_NEAREST)
            _, out_mask = cv2.threshold(out_mask, 127, 255, cv2.THRESH_BINARY)

        return {
            'processed': processed.astype(np.uint8),       # BGR color
            'mask': None if out_mask is None else out_mask.astype(np.uint8),
            'bbox': bbox,
            'metrics': metrics
        }

def _has_cuda() -> bool:
  return hasattr(cv2, "cuda") and cv2.cuda.getCudaEnabledDeviceCount() > 0

class CudaOpenCVPreprocessor(IImagePreprocessor):
    """
    CUDA-accelerated version of your OpenCV pipeline.
    - Keeps BGR throughout.
    - Uses GPU for: blur/median, morphology (Top-hat), color conversions, equalize, unsharp, resize, Canny.
    - Uses CPU (safe + negligible perf hit) for: Otsu, contours/bbox, crop, Laplacian variance, mask resize.
    - Falls back to OpenCVPreprocessor if CUDA not present or any CUDA call fails.
    Returns: {'processed'(BGR uint8), 'mask'(binary uint8 or None), 'bbox', 'metrics'}
    """
    def __init__(self, config: "ProcessingConfig", cpu_fallback: Optional["IImagePreprocessor"]=None):
        self.cfg = config
        # Reuse your existing CPU implementation as fallback
        self.cpu = cpu_fallback or OpenCVPreprocessor(config)
        self.use_cuda = _has_cuda()

    # ---- helpers ----
    def _ensure_odd(self, k: int) -> int:
        return k if k % 2 == 1 else max(1, k - 1)

    def _up(self, arr: np.ndarray) -> "cv2.cuda_GpuMat":
        g = cv2.cuda_GpuMat()
        g.upload(arr)
        return g

    def _down(self, g: "cv2.cuda_GpuMat") -> np.ndarray:
        return g.download()

    def _to_ycrcb_split(self, bgr_gpu: "cv2.cuda_GpuMat"):
        ycrcb_gpu = cv2.cuda.cvtColor(bgr_gpu, cv2.COLOR_BGR2YCrCb)
        Y, Cr, Cb = cv2.cuda.split(ycrcb_gpu)
        return Y, Cr, Cb

    def _merge_ycrcb_to_bgr(self, Y: "cv2.cuda_GpuMat", Cr: "cv2.cuda_GpuMat", Cb: "cv2.cuda_GpuMat"):
        ycrcb_gpu = cv2.cuda.merge((Y, Cr, Cb))
        return cv2.cuda.cvtColor(ycrcb_gpu, cv2.COLOR_YCrCb2BGR)

    def preprocess(self, img_bgr: np.ndarray) -> Dict[str, Any]:
        # Fast/safe exit
        if not self.use_cuda:
            return self.cpu.preprocess(img_bgr)

        cfg = self.cfg
        try:
            # ---- 1) Denoise on color (GPU) ----
            proc_gpu = self._up(img_bgr)
            if cfg.denoise_method == 'gaussian':
                k = self._ensure_odd(cfg.denoise_ksize)
                gf = cv2.cuda.createGaussianFilter(cv2.CV_8UC3, cv2.CV_8UC3, (k, k), cfg.denoise_sigma)
                tmp = cv2.cuda_GpuMat(); gf.apply(proc_gpu, tmp); proc_gpu = tmp
            elif cfg.denoise_method == 'median':
                k = self._ensure_odd(cfg.denoise_ksize)
                mf = cv2.cuda.createMedianFilter(cv2.CV_8UC3, k)
                tmp = cv2.cuda_GpuMat(); mf.apply(proc_gpu, tmp); proc_gpu = tmp

            # ---- 2) Background removal (Top-hat) on luminance (GPU) ----
            if cfg.background_method == 'tophat':
                ksize = max(3, int(cfg.background_se_radius))
                Y, Cr, Cb = self._to_ycrcb_split(proc_gpu)
                morph = cv2.cuda.createMorphologyFilter(cv2.MORPH_TOPHAT, cv2.CV_8UC1, (ksize, ksize))
                top = cv2.cuda_GpuMat(); morph.apply(Y, top)
                Yenh = cv2.cuda.add(Y, top)  # saturates to 255
                proc_gpu = self._merge_ycrcb_to_bgr(Yenh, Cr, Cb)

            # ---- 3) Optional log transform (CPU for numerical stability) ----
            if cfg.log_transform:
                proc_cpu = self._down(proc_gpu).astype(np.float32)
                proc_cpu = np.log1p(proc_cpu * cfg.log_c)
                for c in range(3):
                    ch = proc_cpu[:, :, c]
                    rng = float(np.ptp(ch))
                    if rng < 1e-12:
                        proc_cpu[:, :, c] = 0.0
                    else:
                        ch_min = float(np.min(ch))
                        proc_cpu[:, :, c] = 255.0 * (ch - ch_min) / (rng + 1e-9)
                proc_gpu = self._up(np.clip(proc_cpu, 0, 255).astype(np.uint8))

            # ---- 4) Contrast (eq on luminance; GPU) ----
            ycrcb = cv2.cuda.cvtColor(proc_gpu, cv2.COLOR_BGR2YCrCb)
            Y, Cr, Cb = cv2.cuda.split(ycrcb)
            Yeq = cv2.cuda.equalizeHist(Y)
            ycrcb2 = cv2.cuda.merge((Yeq, Cr, Cb))
            proc_gpu = cv2.cuda.cvtColor(ycrcb2, cv2.COLOR_YCrCb2BGR)

            # ---- 5) Sharpen (unsharp, GPU) ----
            if cfg.sharpen_method == 'unsharp':
                r = self._ensure_odd(cfg.unsharp_radius * 2 + 1)
                gf = cv2.cuda.createGaussianFilter(cv2.CV_8UC3, cv2.CV_8UC3, (r, r), 0)
                blurred = cv2.cuda_GpuMat(); gf.apply(proc_gpu, blurred)
                proc_gpu = cv2.cuda.addWeighted(proc_gpu, 1.0 + cfg.unsharp_amount, blurred, -cfg.unsharp_amount, 0)

            # ---- 6) Mask from luminance (Otsu on CPU) ----
            lum_gpu = cv2.cuda.cvtColor(proc_gpu, cv2.COLOR_BGR2GRAY)
            lum = self._down(lum_gpu)
            mask = None
            if cfg.mask_threshold in ('otsu', 'kapur'):
                _, th = cv2.threshold(lum, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
                mask = th

            # Morphological closing (CPU)
            if mask is not None and cfg.closing_radius > 0:
                kr = max(1, int(cfg.closing_radius))
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kr, kr))
                mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

            # ---- 7) BBox from mask (CPU; fallback full image) ----
            h, w = lum.shape[:2]
            bbox = (0, 0, w, h)
            if mask is not None and mask.sum() > 0:
                contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                if contours:
                    c = max(contours, key=cv2.contourArea)
                    x, y, bw, bh = cv2.boundingRect(c)
                    x = max(0, x - cfg.crop_margin)
                    y = max(0, y - cfg.crop_margin)
                    bw = min(w - x, bw + 2 * cfg.crop_margin)
                    bh = min(h - y, bh + 2 * cfg.crop_margin)
                    bbox = (int(x), int(y), int(bw), int(bh))

            # ---- 8) Crop (CPU) & Resize (GPU) ----
            proc_cpu = self._down(proc_gpu)
            x, y, bw, bh = bbox
            cropped = proc_cpu[y:y+bh, x:x+bw]
            target = max(1, int(cfg.resize_to))
            interp_map = {'bicubic': cv2.INTER_CUBIC, 'bilinear': cv2.INTER_LINEAR, 'nearest': cv2.INTER_NEAREST}
            interp = interp_map.get(cfg.interpolation, cv2.INTER_CUBIC)
            processed_gpu = cv2.cuda.resize(self._up(cropped), (target, target), interpolation=interp)
            processed = self._down(processed_gpu)

            # ---- 9) Metrics (Canny GPU, Laplacian CPU) ----
            lum_proc = cv2.cvtColor(processed, cv2.COLOR_BGR2GRAY)
            canny = cv2.cuda.createCannyEdgeDetector(50, 150)
            edges = self._down(canny.detect(self._up(lum_proc)))
            edge_density = float(edges.sum() / 255) / float(lum_proc.size)
            lap_var = float(cv2.Laplacian(lum_proc, cv2.CV_64F).var())
            metrics = {
                'edge_density': edge_density,
                'laplacian_var': lap_var,
                'orig_shape': (h, w, 3),
                'cropped_shape': processed.shape,
                'quality_pass': True
            }
            if cfg.quality_gate:
                if cfg.min_edge_density is not None and edge_density < cfg.min_edge_density:
                    metrics['quality_pass'] = False
                if cfg.min_laplacian_var is not None and lap_var < cfg.min_laplacian_var:
                    metrics['quality_pass'] = False

            # ---- 10) Resize mask to match output (CPU) ----
            out_mask = None
            if mask is not None:
                mask_cropped = mask[y:y+bh, x:x+bw]
                out_mask = cv2.resize(mask_cropped, (target, target), interpolation=cv2.INTER_NEAREST)
                _, out_mask = cv2.threshold(out_mask, 127, 255, cv2.THRESH_BINARY)

            return {
                'processed': processed.astype(np.uint8),       # BGR color
                'mask': None if out_mask is None else out_mask.astype(np.uint8),
                'bbox': bbox,
                'metrics': metrics
            }

        except Exception:
            # Any CUDA hiccup? Stay robust.
            return self.cpu.preprocess(img_bgr)

In [10]:
# Cell 2: Global Constants and File Paths
# Define constants and paths used throughout the notebook for easy configuration.

#  Paths
# NOTE: Ensure your Google Drive is mounted at /content/drive
BASE_DIR = Path("/content/drive/MyDrive/UP Honours/COS711 - AI/Assignment 3/Data")
RESULTS_DIR = Path("/content/drive/MyDrive/UP Honours/COS711 - AI/Final Results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True) # Create results directory if it doesn't exist

#  Model & Training Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 384
BATCH_SIZE = 16
LEARNING_RATE_BASELINE = 5e-4
LEARNING_RATE_FINETUNE = 1e-4 # Use a lower LR for fine-tuning with pseudo-labels
WEIGHT_DECAY = 1e-4
EPOCHS = 30 # Max epochs; early stopping will likely trigger before this.
PATIENCE = 7 # For early stopping

print(f"Base data directory: {BASE_DIR}")
print(f"Results will be saved to: {RESULTS_DIR}")
print(f"Using device: {DEVICE}")

Base data directory: /content/drive/MyDrive/UP Honours/COS711 - AI/Assignment 3/Data
Results will be saved to: /content/drive/MyDrive/UP Honours/COS711 - AI/Final Results
Using device: cuda


In [11]:
# Cell 3: Data Loading & Initial Exploration
# Load the metadata files (labels.csv, test.csv) and perform an initial count of the image files.

#  Load CSVs
labels_path = BASE_DIR / "labels.csv"
test_path = BASE_DIR / "test.csv"
assert labels_path.exists(), f"'{labels_path}' not found. Please check the path."
assert test_path.exists(), f"'{test_path}' not found. Please check the path."

labels_df_raw = pd.read_csv(labels_path, header=None, names=["ra", "dec", "label", "col4", "col5"])
test_df = pd.read_csv(test_path) # Header is inferred correctly

print(f"labels.csv initial shape: {labels_df_raw.shape}")
display(labels_df_raw.head())
print(f"\ntest.csv shape: {test_df.shape}")
display(test_df.head())

#  Count Image Files
typ_count = len(list((BASE_DIR/"typ").rglob("*.*")))
exo_count = len(list((BASE_DIR/"exo").rglob("*.*")))
unl_count = len(list((BASE_DIR/"unl").rglob("*.*")))
print(f"\nFound {typ_count} typical source files.")
print(f"Found {exo_count} exotic source files.")
print(f"Found {unl_count} unlabelled source files.")

labels.csv initial shape: (2178, 5)


,ra,dec,label,col4,col5
0,10.328221,-20.476357,FR II,NaN,NaN
1,92.109802,-49.431413,typical,NaN,NaN
2,88.916825,-59.431868,Point Source,NaN,NaN
3,5.457981,-25.589637,FR II,NaN,NaN
4,119.417608,-53.396711,FR II,NaN,NaN



test.csv shape: (99, 2)


,201.7436567,-31.32163727
0,234.261286,-46.590846
1,66.793081,-62.375058
2,108.760518,-59.958776
3,202.148240,-31.432391
4,57.025208,-73.861150



Found 2049 typical source files.
Found 72 exotic source files.
Found 13821 unlabelled source files.


In [12]:
# Cell 4: Label Cleaning and Preprocessing
# As per the assignment, we need to handle labels carefully.
# We will NOT remove the 'Should be discarded' class, as the model must be able to identify these images.
# Our preprocessing involves cleaning up unused columns and standardizing the label text format.

# Drop unused columns and rows with missing essential data
labels_df = labels_df_raw[["ra", "dec", "label"]].dropna().copy()

# Standardize label text: title case, strip whitespace, and normalize separators
labels_df["label"] = (
    labels_df["label"]
    .str.strip()
    .str.replace("-", " ", regex=False)
    .str.replace("/", " ", regex=False)
    .str.title() # e.g., 'FR II' -> 'Fr Ii', 'S/Z shaped' -> 'S Z Shaped'
)

print(f"Cleaned labels count: {len(labels_df)}")
print("\nUnique cleaned labels:")
unique_labels = sorted(labels_df["label"].unique())
for label in unique_labels:
    print(f"- {label}")

# We confirm that 'Should Be Discarded' is retained as a valid class.
assert 'Should Be Discarded' in unique_labels

Cleaned labels count: 2178

Unique cleaned labels:
- Bent
- Exotic
- Fr I
- Fr Ii
- Point Source
- S Z Shaped
- Should Be Discarded
- Typical
- X Shaped


In [13]:
# Cell 5: Image File Discovery and Coordinate Extraction
# We need to map labels to images via coordinates.
# First, we'll recursively find all image files and extract coordinates from their filenames.
# The filenames contain coordinates, but they are not always an exact match to the label coordinates.

def extract_coords_from_filename(fname):
    """Extracts the first two floating-point numbers (RA, Dec) from a filename string."""
    name = Path(fname).stem
    # Regex to find floating-point numbers, including negative ones
    nums = re.findall(r"-?\d+\.\d+", name)
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    return None # Return None if coords can't be found

# Find all image files in 'typ', 'exo', and 'unl' directories
all_images = []
for subset in ["typ", "exo", "unl"]:
    subset_path = BASE_DIR / subset
    for f in subset_path.rglob("*"):
        if f.suffix.lower() in [".png", ".jpg", ".jpeg"] and f.is_file():
            coords = extract_coords_from_filename(f.name)
            if coords: # Only include images where coordinates could be extracted
                all_images.append({
                    "filepath": str(f),
                    "subset": subset,
                    "coords": coords
                })

images_df = pd.DataFrame(all_images)
print(f"Found {len(images_df)} total images with extractable coordinates.")
# print(f"From {typ_count + exo_count + unl_count} we matched {len(images_df)} images.\n We couldn't get {(typ_count + exo_count + unl_count) - {len(images_df)}} images via cordinates")
display(images_df.head())

Found 15941 total images with extractable coordinates.


,filepath,subset,coords
0,/content/drive/MyDrive/UP Honours/COS711 - AI/...,typ,"(49.153, -44.155)"
1,/content/drive/MyDrive/UP Honours/COS711 - AI/...,typ,"(124.739, -56.769)"
2,/content/drive/MyDrive/UP Honours/COS711 - AI/...,typ,"(342.905, -16.207)"
3,/content/drive/MyDrive/UP Honours/COS711 - AI/...,typ,"(354.441, -9.241)"
4,/content/drive/MyDrive/UP Honours/COS711 - AI/...,typ,"(355.396, -8.504)"


In [14]:
# Cell 5.5: Image Preprocessing Pipeline, see it's definition under src/

# # Configure processing
cfg = ProcessingConfig()
cfg.resize_to = 369 # Keep original image size

repo = FileSystemImageRepository()
pre = CudaOpenCVPreprocessor(cfg)

# Run preprocessing on the base dataset directory and save into a `processed` subfolder
OUT_SUBDIR = 'processed'
print(f"Running project preprocessing pipeline on {BASE_DIR} -> {OUT_SUBDIR}")

preprocess_dataset(Path(BASE_DIR), OUT_SUBDIR, repo, pre, cfg, patterns=["*.png"])

# Update images_df to point to the processed images when available
processed_root = Path(BASE_DIR) / 'processed' # Assuming 'processed' is the output subdirectory
print(f"Attempting to update image paths to use processed images from: {processed_root}")

def map_to_processed(path_str):
    p = Path(path_str)
    try:
        rel = p.relative_to(Path(BASE_DIR))
    except Exception:
        return str(p)
    candidate = processed_root / rel.with_suffix('.png')
    return str(candidate) if candidate.exists() else str(p)

images_df['filepath'] = images_df['filepath'].apply(map_to_processed)
print("Done updating filepaths. Sample processed path:", images_df.iloc[0]['filepath'])

# Quick verification
sample = Path(images_df.iloc[0]['filepath'])
print(f"Exists: {sample.exists()} | Path: {sample}")

print('\n✅ Assuming preprocessing has been done and processed images are in a "processed" subdirectory. Filepaths updated in images_df.')

Running project preprocessing pipeline on /content/drive/MyDrive/UP Honours/COS711 - AI/Assignment 3/Data -> processed
Found 15941 images under /content/drive/MyDrive/UP Honours/COS711 - AI/Assignment 3/Data


KeyboardInterrupt: 

In [ ]:
# Cell 6: Coordinate Matching using k-d Tree
# To solve the inexact coordinate matching problem, we use a k-d tree.
# This is a highly efficient data structure for finding the nearest neighbor in a multi-dimensional space.
# We build a tree from all image coordinates and query it with each label's coordinates to find the closest image file.

# Separate labeled images (typ, exo) from unlabeled (unl)
labeled_images_df = images_df[images_df['subset'].isin(['typ', 'exo'])].copy()

# Build the k-d tree on the coordinates of labeled images
image_coords = np.array(labeled_images_df['coords'].tolist())
kdtree = cKDTree(image_coords)

# For each label, find the closest image in the k-d tree
label_coords = labels_df[['ra', 'dec']].values
distances, indices = kdtree.query(label_coords, k=1)

# Assign the matched filepath to each label
labels_df['filepath'] = labeled_images_df.iloc[indices]['filepath'].values

print("Matched labels to their closest image files.")
display(labels_df[['ra', 'dec', 'label', 'filepath']].head())

In [ ]:
# Cell 7: Final DataFrame Preparation and Multi-Label Encoding
# The problem is multi-label, as some sources have multiple classifications (e.g., 'FR I' and 'Bent')[cite: 48].
# We group the labels by filepath and then use MultiLabelBinarizer to create a one-hot encoded vector for each image.

# Group labels by the matched filepath
grouped_labels = labels_df.groupby('filepath')['label'].apply(list).reset_index()

# Merge the grouped labels back into the main labeled images dataframe
# We use a 'left' merge to keep all images, even those with no matching label (they will get an empty list).
merged_df = pd.merge(labeled_images_df, grouped_labels, on='filepath', how='left')
# Fill NaN labels (images with no match) with an empty list for the binarizer
merged_df['label'] = merged_df['label'].apply(lambda x: x if isinstance(x, list) else [])

#  Multi-Label Binarization
mlb = MultiLabelBinarizer()
multi_hot_labels = mlb.fit_transform(merged_df['label'])
merged_df['multi_hot'] = list(multi_hot_labels)

# Save the classes found by the binarizer for later use
LABEL_CLASSES = mlb.classes_.tolist()

print(f"DataFrame prepared for training with {len(merged_df)} labeled images.")
print(f"Found {len(LABEL_CLASSES)} unique classes:")
print(LABEL_CLASSES)
display(merged_df[['filepath', 'label', 'multi_hot']].head())

In [ ]:
# Cell 8: Addressing Class Imbalance with Weighted Loss
# The dataset is highly imbalanced, with many 'FR I/II' sources and very few 'X-Shaped' ones.
# To prevent the model from ignoring rare classes, we will use a weighted loss function.
# The weight for each class is the inverse of its frequency, encouraging the model to pay more attention to under-represented classes.

class_counts = merged_df['multi_hot'].sum(axis=0)
class_frequencies = class_counts / len(merged_df)

# Calculate positive weights for BCEWithLogitsLoss
pos_weight = (len(merged_df) - class_counts) / (class_counts + 1e-9)
pos_weight = np.clip(pos_weight, None, 30.0)
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(DEVICE)

# Define the weighted loss function
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

# Visualize Class Distribution
plt.figure(figsize=(12, 6))
sns.barplot(x=LABEL_CLASSES, y=class_counts)
plt.title('Class Distribution in Labeled Dataset')
plt.ylabel('Number of Images')
plt.xticks(rotation=45, ha='right')
plt.show()

print("Calculated positive weights for BCEWithLogitsLoss to handle class imbalance:")
for cls, weight in zip(LABEL_CLASSES, pos_weight):
    print(f"- {cls}: {weight:.2f}")


In [ ]:
# Cell 9: Dataset, Data Augmentation, and DataLoaders
# We define advanced data augmentations to artificially expand the small training set and improve model generalization[cite: 31, 63].
# We then create PyTorch Dataset and DataLoader objects to feed data to the model efficiently.

# Data Augmentation Transforms
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.1),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7,1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# PyTorch Dataset Class
class RadioDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['filepath']
        try:
            img = Image.open(img_path).convert('RGB')
        except (UnidentifiedImageError, OSError):
            # If an image is corrupt, load the next one instead.
            return self.__getitem__((idx + 1) % len(self.df))

        if self.transform:
            img = self.transform(img)

        target = torch.tensor(row['multi_hot'], dtype=torch.float32)
        return img, target

# Train/Validation Split
train_df, val_df = train_test_split(merged_df, test_size=0.15, random_state=SEED)
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")

# Create DataLoaders
train_dataset = RadioDataset(train_df, transform=train_transforms)
val_dataset = RadioDataset(val_df, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# Test one batch to ensure it works
imgs, targets = next(iter(train_loader))
print(f"\nBatch of images shape: {imgs.shape}")
print(f"Batch of targets shape: {targets.shape}")

In [ ]:
# Cell 10: Model Architecture Definition
# We define functions to get two powerful pre-trained models: EfficientNetV2-S and EfficientNetV2-M
# We will compare their performance to select the best one for our task.
# The final classification layer of each model is replaced to match our number of classes.

def get_efficientnet_v2(n_classes):
    """Loads a pre-trained EfficientNetV2-S and adapts its classifier."""
    try:
        model = models.efficientnet_v2_s(weights="IMAGENET1K_V1")
    except TypeError: # Fallback for older torchvision
        model = models.efficientnet_v2_s(pretrained=True)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, n_classes)
    return model.to(DEVICE)

def get_efficientnetv2(n_classes):
    """Loads a pre-trained EfficientNetV2-M and adapts its classifier."""
    try:
        model = models.efficientnet_v2_m(weights='IMAGENET1K_V1')
    except TypeError:
        model = models.efficientnet_v2_m(pretrained=True)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, n_classes)
    return model.to(DEVICE)

# Early Stopping Utility
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_metric = None
        self.counter = 0

    def step(self, metric):
        if self.best_metric is None or metric > self.best_metric + self.min_delta:
            self.best_metric = metric
            self.counter = 0
            return False # Continue training
        else:
            self.counter += 1
            return self.counter >= self.patience # Stop training

In [ ]:
# Cell 11: Training, Evaluation, and Optimization Utilities

# Main Evaluation Function
def evaluate_model(loader, model, criterion, device=DEVICE, threshold=0.5, return_details=False):
    """Evaluates the model on a given dataset loader, returning loss and several metrics."""
    model.eval()
    total_loss = 0.0
    all_logits = []
    all_targets = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            if isinstance(y, torch.Tensor):
                y = y.to(device)
            else:
                y = torch.tensor(y, device=device)

            logits = model(x)

            if isinstance(criterion, (nn.BCEWithLogitsLoss, LabelSmoothingBCEWithLogitsLoss)):
                y_for_loss = y.float()
            else:
                y_for_loss = y.long()

            loss = criterion(logits, y_for_loss)
            total_loss += float(loss.detach().cpu()) * x.size(0)

            all_logits.append(logits.cpu())
            all_targets.append(y.cpu())

    avg_loss = total_loss / len(loader.dataset)
    all_logits = torch.cat(all_logits)
    all_targets = torch.cat(all_targets)

    if isinstance(criterion, (nn.BCEWithLogitsLoss, LabelSmoothingBCEWithLogitsLoss)):
        probs = torch.sigmoid(all_logits)
        # Allow for per-class thresholds
        preds = (probs >= torch.tensor(threshold, dtype=torch.float32)).int()
        true = all_targets.int()
        per_class_f1 = f1_score(true.numpy(), preds.numpy(), average=None, zero_division=0)
        macro_f1 = f1_score(true.numpy(), preds.numpy(), average='macro', zero_division=0)
        micro_f1 = f1_score(true.numpy(), preds.numpy(), average='micro', zero_division=0)
        exact_match = (preds == true).all(dim=1).float().mean().item()
        metrics = {
            'val_loss': avg_loss,
            'macro_f1': float(macro_f1),
            'micro_f1': float(micro_f1),
            'exact_match': float(exact_match),
            'per_class_f1': per_class_f1
        }
    else: # Single-label path (remains for flexibility)
        probs = torch.softmax(all_logits, dim=1)
        preds = probs.argmax(dim=1)
        true = all_targets.long()
        acc = (preds == true).float().mean().item()
        macro_f1 = f1_score(true.numpy(), preds.numpy(), average='macro', zero_division=0)
        metrics = {'val_loss': avg_loss, 'accuracy': float(acc), 'macro_f1': float(macro_f1)}

    if return_details:
        return metrics, preds, true, probs
    return metrics

# Post-Training Threshold Optimization
def find_best_thresholds(model, val_loader, device=DEVICE):
    """Finds the optimal per-class threshold to maximize F1-score on the validation set."""
    model.eval()
    all_probs = []
    all_targets = []
    with torch.no_grad():
        for imgs, targets in val_loader:
            logits = model(imgs.to(device))
            probs = torch.sigmoid(logits).cpu()
            all_probs.append(probs)
            all_targets.append(targets)

    all_probs = torch.cat(all_probs).numpy()
    all_targets = torch.cat(all_targets).numpy()

    best_thresholds = np.zeros(all_probs.shape[1])
    best_f1s = np.zeros(all_probs.shape[1])

    print("🚀 Finding optimal threshold for each class...")
    for i in range(all_probs.shape[1]): # Iterate over each class
        thresholds = np.linspace(0.05, 0.95, 100)
        f1_scores = [f1_score(all_targets[:, i], (all_probs[:, i] >= t).astype(int), zero_division=0) for t in thresholds]
        best_t_idx = np.argmax(f1_scores)
        best_thresholds[i] = thresholds[best_t_idx]
        best_f1s[i] = f1_scores[best_t_idx]
        print(f"- Class '{LABEL_CLASSES[i]}': Best Threshold = {best_thresholds[i]:.2f} (F1 = {best_f1s[i]:.4f})")

    return best_thresholds

# Regularization Helper Functions
def mixup_data(x, y, alpha=0.4, device='cuda'):
    if alpha > 0: lam = np.random.beta(alpha, alpha)
    else: lam = 1
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Main Training Function (with OneCycleLR)
def train_model(model, train_loader, val_loader,
                epochs=25, head_epochs=5, head_lr=1e-3, finetune_max_lr=3e-4,
                weight_decay=1e-3, patience=8, device=DEVICE, grad_clip=1.0,
                model_save_path='best_model.pth',
                use_label_smoothing=False, smoothing_factor=0.1,
                use_mixup=False, mixup_alpha=0.4,
                accumulation_steps=1):

    model = model.to(device)

    # Loss Function Setup
    if use_label_smoothing:
        print(f"Using Label Smoothing with factor {smoothing_factor}")
        criterion = LabelSmoothingBCEWithLogitsLoss(smoothing=smoothing_factor).to(device)
    else:
        class_counts = train_df['multi_hot'].sum(axis=0)
        pos_weight = (len(train_df) - class_counts) / (class_counts + 1e-9)
        pos_weight = np.clip(pos_weight, None, 30.0)
        pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

    scaler = GradScaler()
    history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
    stopper = EarlyStopping(patience=patience)
    best_val_f1 = -1.0

    def _train_one_epoch(current_optimizer, scheduler=None, use_mixup_epoch=False):
        model.train()
        running_loss = 0.0
        n_samples = 0
        pbar = progress_bar(train_loader, parent=None)
        current_optimizer.zero_grad()

        for i, (xb, yb) in enumerate(pbar):
            xb, yb = xb.to(device), yb.to(device).float()

            with autocast():
                if use_mixup_epoch:
                    mixed_xb, yb_a, yb_b, lam = mixup_data(xb, yb, alpha=mixup_alpha, device=device)
                    logits = model(mixed_xb)
                    loss = mixup_criterion(criterion, logits, yb_a, yb_b, lam)
                else:
                    logits = model(xb)
                    loss = criterion(logits, yb)

            loss = loss / accumulation_steps
            scaler.scale(loss).backward()

            if (i + 1) % accumulation_steps == 0:
                scaler.unscale_(current_optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(current_optimizer)
                scaler.update()
                if scheduler:
                    scheduler.step()
                current_optimizer.zero_grad()

            running_loss += float(loss.detach().cpu()) * xb.size(0) * accumulation_steps
            n_samples += xb.size(0)
        return running_loss / n_samples

    # Phase A: Head Training
    print("🧠 Starting Head Training ")
    try:
        for n, p in model.named_parameters(): p.requires_grad = False
        for p in model.classifier.parameters(): p.requires_grad = True
    except:
        for n, p in model.named_parameters(): p.requires_grad = 'classifier' in n or 'fc' in n or 'head' in n

    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(head_params, lr=head_lr, weight_decay=weight_decay)

    for e in range(head_epochs):
        train_loss = _train_one_epoch(optimizer, scheduler=None, use_mixup_epoch=False)
        val_metrics = evaluate_model(val_loader, model, criterion, device=device)
        val_f1 = val_metrics.get('macro_f1', 0.0)
        print(f"[Head] Epoch {e+1}/{head_epochs} | train_loss={train_loss:.4f} | val_macro_f1={val_f1:.4f} | val_loss={val_metrics['val_loss']:.4f}")

    # Phase B: Fine-tuning
    print("\n Starting Full Fine-Tuning ")
    for p in model.parameters(): p.requires_grad = True
    optimizer = AdamW(model.parameters(), lr=finetune_max_lr, weight_decay=weight_decay) # LR is the max_lr for OneCycle

    total_ft_epochs = max(1, epochs - head_epochs)

    scheduler = OneCycleLR(optimizer, max_lr=finetune_max_lr,
                           steps_per_epoch=(len(train_loader) // accumulation_steps),
                           epochs=total_ft_epochs, pct_start=0.3)

    for e in range(total_ft_epochs):
        train_loss = _train_one_epoch(optimizer, scheduler=scheduler, use_mixup_epoch=use_mixup)
        val_metrics = evaluate_model(val_loader, model, criterion, device=device)
        val_f1 = val_metrics.get('macro_f1', 0.0)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['val_loss'])
        history['val_macro_f1'].append(val_f1)

        print(f"[Finetune] Epoch {e+1}/{total_ft_epochs} | train_loss={train_loss:.4f} | val_macro_f1={val_f1:.4f} | val_loss={val_metrics['val_loss']:.4f}")

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), model_save_path)
            print(f"   >>> 💾 New best model saved with val_macro_f1: {best_val_f1:.4f} <<<")

        if stopper.step(val_f1):
            print("🛑 Early stopping triggered during finetuning.")
            break

    return model, history

In [ ]:
import torch.nn as nn
import numpy as np

class LabelSmoothingBCEWithLogitsLoss(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingBCEWithLogitsLoss, self).__init__()
        self.smoothing = smoothing
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, inputs, targets):
        # Apply label smoothing
        targets_smooth = targets * (1.0 - self.smoothing) + self.smoothing / targets.size(1)

        # Get the pos_weight from the original criterion if it exists
        # This assumes 'criterion' is in the global scope from the setup cell
        try:
            pos_weight = criterion.pos_weight
            loss = self.bce(inputs, targets_smooth)
            # Manually apply pos_weight
            loss = loss * (pos_weight * targets + (1 - targets))
        except (NameError, AttributeError):
            # Fallback if original criterion or pos_weight isn't available
            loss = self.bce(inputs, targets_smooth)

        return loss.mean()

In [ ]:
# Sanity check just to see if our dataset pipeline and shares are good to train the model
model = get_efficientnetv2(len(LABEL_CLASSES)) # or call your get_efficientnetv2
print("Device (first param):", next(model.parameters()).device)
print("Total params:", sum(p.numel() for p in model.parameters()))
print("Trainable params (before freeze):", sum(p.numel() for p in model.parameters() if p.requires_grad))

# Take one batch
xb, yb = next(iter(train_loader))
print("xb shape/dtype/min/max:", xb.shape, xb.dtype, xb.min().item(), xb.max().item())
print("yb shape/dtype/min/max:", yb.shape, yb.dtype, yb.min().item(), yb.max().item())

# Forward + loss test
model.eval()
with torch.no_grad():
    out = model(xb.to(DEVICE))
print("out shape:", out.shape)

# check BCE loss (casts)
try:
    l = nn.BCEWithLogitsLoss()(out, yb.float().to(DEVICE))
    print("BCEWithLogitsLoss computed OK:", float(l))
except Exception as e:
    print("BCE error:", e)

# quick grad flow test (head-only)
for n,p in model.named_parameters():
    p.requires_grad = False
# enable classifier params
for p in model.classifier.parameters():
    p.requires_grad = True

opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)
model.train()
opt.zero_grad()
out = model(xb.to(DEVICE))
loss = nn.BCEWithLogitsLoss()(out, yb.float().to(DEVICE))
loss.backward()
print("Grad for classifier first param is None?:", next(p.grad for p in model.classifier.parameters()) is None)


## Baseline Training

In [ ]:
# Cell 12: Execute Focused Baseline Training Run & Final Evaluation

# 1. Define Hyperparameters from Best Optuna Trial
# Sourced from your best trial results
best_params = {
    'finetune_max_lr': 2.5578e-4,
    'weight_decay': 1.3506e-4,
    'mixup_alpha': 0.1143,
    'smoothing_factor': 0.1668
}
print("🚀 Starting focused training run with the following hyperparameters:")
for key, value in best_params.items():
    print(f"  - {key}: {value}")

# 2. Initialize Model
baseline_model = get_efficientnetv2(len(LABEL_CLASSES))
baseline_model_path = RESULTS_DIR / 'best_baseline_model_onecycle.pth'
/content/drive/MyDrive/UP Honours/COS711 - AI/Assignment 3/Final Results/best_baseline_model_onecycle.pth

# 3. Execute the Training Run
# This will use the new train_model function with OneCycleLR
baseline_model, baseline_history = train_model(
    model=baseline_model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=40, # Train for a longer duration; early stopping will handle the rest
    head_epochs=5,
    head_lr=1e-3,
    finetune_max_lr=best_params['finetune_max_lr'],
    weight_decay=best_params['weight_decay'],
    patience=8, # A reasonable patience for a full run
    model_save_path=baseline_model_path,
    use_label_smoothing=True,
    smoothing_factor=best_params['smoothing_factor'],
    use_mixup=True,
    mixup_alpha=best_params['mixup_alpha'],
    accumulation_steps=4
)

print(f"\n Training complete. Best baseline model saved to {baseline_model_path}")

In [ ]:
# Cell 13:  Baseline's Final Evaluation
# 4. Post-Training Optimization and Final Evaluation
print("\nEvaluating Final Baseline Model with Optimized Thresholds ")

# Load the best performing weights from the training run
final_model = get_efficientnetv2(len(LABEL_CLASSES))
final_model.load_state_dict(torch.load(baseline_model_path))
final_model.to(DEVICE)

# Find the optimal per-class thresholds on the validation set
best_thresholds = find_best_thresholds(final_model, val_loader)

# Create a dummy criterion instance for the evaluation function.
# Its parameters don't affect metric calculation, only the loss value.
final_criterion = nn.BCEWithLogitsLoss()

# Re-evaluate using the newly found best thresholds
final_metrics = evaluate_model(
    val_loader,
    final_model,
    final_criterion,
    device=DEVICE,
    threshold=best_thresholds # Pass the array of optimal thresholds
)

print("\n Final Baseline Performance ")
print(f"  - Validation Loss: {final_metrics['val_loss']:.4f}")
print(f"  - Macro F1-Score (Optimized Thresholds): {final_metrics['macro_f1']:.4f}")
print(f"  - Micro F1-Score (Optimized Thresholds): {final_metrics['micro_f1']:.4f}")
print(f"  - Exact Match Ratio: {final_metrics['exact_match']:.4f}")

## Psudo-Labelling

The pseudo-labeling stage is where we use our well-trained baseline model as a teacher to expand the training data and strengthen the final model. The idea is to let the model label the previously unlabelled images, but only keep the most confident predictions as “pseudo-labels.”

Goal: To create a richer, more balanced training set by adding reliable model-generated labels, especially for rare or underrepresented classes. Key steps:

1. Generating Predictions: By running inference on all 13,821 unlabelled images.

2. Filtering High-Confidence Samples: Implementing a robust Top-K filtering strategy to select the most promising candidates for each class, ensuring our rare classes get represented.

4. Combining the Datasets: By merging the original labeled data with the new pseudo-labeled samples to form an enhanced training set.

5. Lastly, we retrain the final model: By initializing a fresh EfficientNetV2-M model with ImageNet weights, then train it on the combined dataset using the same setup (OneCycleLR, weighted loss, etc.). We'll use a slightly lower finetune_max_lr (e.g., 1e-4) to ensure stable learning since pseudo-labels can introduce mild noise.


In [ ]:
# Cell 13: Phase 2 - Pseudo-Labeling Pipeline

print(" Starting Phase 2: Pseudo-Labeling Pipeline ")

# 1. Load Best Baseline Model
# Ensure 'best_thresholds' is in memory from the previous cell's execution
assert 'best_thresholds' in locals(), "Error: 'best_thresholds' not found. Please re-run the previous cell."

print("Loading best baseline model for inference...")
baseline_model = get_efficientnetv2(len(LABEL_CLASSES))
baseline_model_path = Path('/content/drive/MyDrive/UP Honours/COS711 - AI/Assignment 3/Final Results/best_baseline_model_onecycle.pth')
assert baseline_model_path.exists(), f"Error: Model file not found at {baseline_model_path}"

baseline_model.load_state_dict(torch.load(baseline_model_path))
baseline_model.to(DEVICE)
baseline_model.eval()

# 2. Prepare Unlabeled Data Loader
print("Preparing DataLoader for 13,821 unlabeled images...")
unlabeled_df = images_df[images_df['subset'] == 'unl'].copy().reset_index(drop=True)
# Add a dummy 'multi_hot' column to be compatible with RadioDataset
unlabeled_df['multi_hot'] = [[0] * len(LABEL_CLASSES)] * len(unlabeled_df)

unlabeled_dataset = RadioDataset(unlabeled_df, transform=val_transforms)
# Use a larger batch size for faster inference
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

# 3. Run Inference on All Unlabeled Data
print(f"Running inference on {len(unlabeled_df)} unlabeled images...")
all_probs = []
with torch.no_grad():
    for imgs, _ in progress_bar(unlabeled_loader, parent=None):
        logits = baseline_model(imgs.to(DEVICE))
        probs = torch.sigmoid(logits).cpu()
        all_probs.append(probs)

unlabeled_probs = torch.cat(all_probs).numpy()
print(" Inference complete.")

# Store probabilities in the unlabeled dataframe for filtering
for i, cls in enumerate(LABEL_CLASSES):
    unlabeled_df[cls] = unlabeled_probs[:, i]

In [ ]:
# Cell: 14 - Pseudo-Labeling Pipeline continued

# 4. Filter for High-Confidence Pseudo-Labels (Top-K Strategy)
# We select the Top K most confident predictions for EACH class.
# This ensures we get high-quality candidates even for rare classes.
K = 200 # Select top 200 candidates per class
print(f"Filtering pseudo-labels using Top-{K} per-class strategy...")

pseudo_labeled_indices = set()
for cls in LABEL_CLASSES:
    # Get the indices of the K highest-probability samples for this class
    top_k_indices = unlabeled_df[cls].nlargest(K).index
    pseudo_labeled_indices.update(top_k_indices)

# Create the initial pseudo-label dataframe
pseudo_df = unlabeled_df.loc[list(pseudo_labeled_indices)].copy()
original_pseudo_count = len(pseudo_df)
print(f"Selected {original_pseudo_count} unique candidates.")

# 5. Apply Optimized Thresholds to Generate Final Labels
# This is the most critical step. We use our per-class thresholds,
# not a generic 0.5, to assign the final binary labels.
print(f"Applying optimized 'best_thresholds' to assign labels...")
pseudo_probs_selected = pseudo_df[LABEL_CLASSES].values
pseudo_multi_hot = (pseudo_probs_selected >= best_thresholds).astype(int)
pseudo_df['multi_hot'] = list(pseudo_multi_hot)

# Important: Filter out any samples that, after thresholding, have NO labels.
# These are low-quality candidates we must discard.
pseudo_df = pseudo_df[pseudo_df['multi_hot'].apply(sum) > 0]
final_pseudo_count = len(pseudo_df)

print(f"Kept {final_pseudo_count} high-quality samples after thresholding and filtering empty labels.")

# 6. Create Final Combined Dataset & DataLoader
print("\nCombining Datasets")
# 'train_df' is the original human-labeled training set from Cell 9
combined_train_df = pd.concat([train_df, pseudo_df], ignore_index=True)

print(f"Original human-labeled training size: {len(train_df)}")
print(f"Added {len(pseudo_df)} high-quality pseudo-labeled samples.")
print(f"Total combined training size for final model: {len(combined_train_df)}")

print("\nCreating new DataLoaders for final training...")
combined_train_dataset = RadioDataset(combined_train_df, transform=train_transforms)
combined_train_loader = DataLoader(combined_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
# Note: val_loader remains unchanged.

print("\nCompleted successfully.")

In [ ]:
# Cell 15: Save Key Artifacts for Future Sessions

import pandas as pd
import numpy as np
import torch
from pathlib import Path

print("Saving Pipeline Artifacts")

# Define a dedicated directory to store these artifacts
ARTIFACTS_DIR = RESULTS_DIR / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)
print(f"Artifacts will be saved in: {ARTIFACTS_DIR}")

# 1. Save the final pseudo-labeled dataframe
# This contains the high-confidence samples selected for training.
pseudo_df_path = ARTIFACTS_DIR / 'pseudo_df.parquet'
pseudo_df.to_parquet(pseudo_df_path)
print(f"Saved pseudo-labeled dataframe to {pseudo_df_path}")

# 2. Save the final combined training dataframe
# This is the original training set plus the new pseudo-labeled data.
combined_df_path = ARTIFACTS_DIR / 'combined_train_df.parquet'
combined_train_df.to_parquet(combined_df_path)
print(f"Saved combined training dataframe to {combined_df_path}")

# 3. Save the raw probabilities of the entire unlabeled set
# This allows you to experiment with different filtering/thresholding strategies
# (e.g., changing K or thresholds) in the future without re-running model inference.
unlabeled_probs_path = ARTIFACTS_DIR / 'unlabeled_probs.npy'
np.save(unlabeled_probs_path, unlabeled_probs)
print(f"Saved unlabeled set probabilities to {unlabeled_probs_path}")

print("\nAll artifacts saved successfully!")

1.Load pre-saved artifacts

In [ ]:
print("Loading Pre-Generated Artifacts")

# 1. Define Paths
# Ensure these variables are defined in your setup cells:
# RESULTS_DIR, BATCH_SIZE, train_transforms, val_transforms, val_df
ARTIFACTS_DIR = ARTIFACTS_DIR
combined_df_path = Path(f'{ARTIFACTS_DIR}/combined_train_df.parquet')

assert combined_df_path.exists(), f"Error: File not found at {combined_df_path}. Please generate artifacts first."

# 2. Load the Combined Training DataFrame
print(f"Loading combined training data from {combined_df_path}...")
combined_train_df = pd.read_parquet(combined_df_path)
# The 'multi_hot' column is correctly loaded as a list of lists from the parquet format.
print(f"Successfully loaded {len(combined_train_df)} training samples.")

# 3. Recreate the DataLoaders
# You can now create the DataLoaders needed for the final training phase.
print("\nRecreating DataLoaders for final training...")

# Recreate the combined training dataset and loader
combined_train_dataset = RadioDataset(combined_train_df, transform=train_transforms)
combined_train_loader = DataLoader(combined_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
print("Created 'combined_train_loader'.")

# Recreate the validation loader (assuming 'val_df' is available from your initial data split)
val_dataset = RadioDataset(val_df, transform=val_transforms)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)
print("Created 'val_loader'.")

print("\nEnvironment is ready for final model training")

In [ ]:
# Implement the Quality Control Filter
# First we take the loaded combined_train_df, remove the flagged samples, and create the final, clean DataLoader we need for training.

# Cell 14: Final Data Preparation & Quality Control

print("Loading Artifacts and Applying Quality Control")

# 1. Load Saved DataFrames

combined_train_df_path = combined_df_path
assert combined_train_df_path.exists(), "Combined training df not found!"

combined_train_df = pd.read_parquet(combined_train_df_path)
print(f"Loaded combined training dataframe with {len(combined_train_df)} samples.")

# 2. Implement the "Should be Discarded" Filter
# Find the index for the 'Should Be Discarded' class
discard_class_index = LABEL_CLASSES.index('Should Be Discarded')
print(f"Filtering out samples flagged as 'Should Be Discarded' (Class Index: {discard_class_index})...")

initial_count = len(combined_train_df)

# Keep only rows where the 'Should Be Discarded' label is 0
combined_train_df = combined_train_df[combined_train_df['multi_hot'].apply(lambda x: x[discard_class_index] == 0)]

final_count = len(combined_train_df)
num_removed = initial_count - final_count
print(f"Removed {num_removed} samples flagged as poor quality.")
print(f"Final, cleaned training set size: {final_count}")

# 3. Create Final, Cleaned DataLoaders
print("\nCreating final, cleaned DataLoaders...")
final_train_dataset = RadioDataset(combined_train_df, transform=train_transforms)
final_train_loader = DataLoader(final_train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
# The validation loader remains the same, as we always evaluate on the original, untouched validation set.
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("\nData preparation complete. Ready for final training.")

## Final Model Training

In this phase, we train a new EfficientNetV2-M model on our larger, cleaner dataset. This is the final model we use to generate labels for our submission.

In [ ]:
# Cell 15: Phase 3 - Final Model Training

print("Starting Phase 3: Final Model Training")

# 1. Initialize a Fresh Model
# We start with new ImageNet weights to learn from our enhanced dataset.
final_model = get_efficientnetv2(len(LABEL_CLASSES))
final_model_path = RESULTS_DIR / 'best_final_enhanced_model.pth'

# 2. Define Hyperparameters
# Using a slightly more conservative learning rate for this larger, pseudo-labeled dataset.
final_params = {
    'finetune_max_lr': 1.5e-4,
    'weight_decay': 1.3506e-4,
    'mixup_alpha': 0.1143,
    'smoothing_factor': 0.1668
}
print("Training with the following hyperparameters:")
for key, value in final_params.items():
    print(f"  - {key}: {value}")

# 3. Execute the Final Training
final_model, final_history = train_model(
    model=final_model,
    train_loader=final_train_loader,
    val_loader=val_loader,
    epochs=40,
    head_epochs=5,
    head_lr=1e-3,
    finetune_max_lr=final_params['finetune_max_lr'],
    weight_decay=final_params['weight_decay'],
    patience=8,
    model_save_path=final_model_path,
    use_label_smoothing=True,
    smoothing_factor=final_params['smoothing_factor'],
    use_mixup=True,
    mixup_alpha=final_params['mixup_alpha'],
    accumulation_steps=4
)

print(f"\n Final training complete. Best enhanced model saved to {final_model_path}")

## Final Evaluation with Optimized Thresholds
We load the trained final model, find its unique optimal thresholds, to get our final performance score.

In [ ]:
# Cell 16: Final Model Evaluation with Optimized Thresholds

print("Starting Final Evaluation")

# 1. Load the Best Enhanced Model
final_model_path = RESULTS_DIR / 'best_final_enhanced_model.pth'
assert final_model_path.exists(), f"Error: Final model not found at {final_model_path}"

final_model = get_efficientnetv2(len(LABEL_CLASSES))
final_model.load_state_dict(torch.load(final_model_path))
final_model.to(DEVICE)
print(f"Successfully loaded final enhanced model from {final_model_path}")

# 2. Find the Optimal Thresholds for THIS Model
# Each model learns different probability distributions, so we must re-run this.
final_best_thresholds = find_best_thresholds(final_model, val_loader)

# 3. Calculate Final, Optimized Performance Metrics
# Create a dummy criterion for the evaluation function call
final_criterion = nn.BCEWithLogitsLoss()

final_metrics = evaluate_model(
    val_loader,
    final_model,
    final_criterion,
    device=DEVICE,
    threshold=final_best_thresholds
)

print("\n Final ENHANCED Model Performance ")
print(f"  - Validation Loss: {final_metrics['val_loss']:.4f}")
print(f"  - Macro F1-Score (Optimized): {final_metrics['macro_f1']:.4f}")
print(f"  - Micro F1-Score (Optimized): {final_metrics['micro_f1']:.4f}")
print(f"  - Exact Match Ratio: {final_metrics['exact_match']:.4f}")

## Submission Files

In [ ]:
# Cell 17: Generate Final Submission Deliverables

print("Generating Final Submission Files")
# Ensure the final model and its optimized thresholds are in memory
assert 'final_model' in locals(), "Final model not found. Please run the evaluation cell first."
assert 'final_best_thresholds' in locals(), "Final best thresholds not found. Please run the evaluation cell first."

final_model.eval()

# 1. Generate `test_labels.csv`
print("\nGenerating predictions for test_labels.csv...")

# We need to map test coordinates to the full set of available image files
full_image_coords = np.array(images_df['coords'].tolist())
full_kdtree = cKDTree(full_image_coords)

test_coords = test_df.iloc[:, [0, 1]].values
distances, indices = full_kdtree.query(test_coords, k=1)

test_pred_df = test_df.copy()
test_pred_df['filepath'] = images_df.iloc[indices]['filepath'].values
test_pred_df['multi_hot'] = [[0]*len(LABEL_CLASSES)] * len(test_pred_df) # Dummy labels

test_dataset = RadioDataset(test_pred_df, transform=val_transforms)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2)

test_probs = []
with torch.no_grad():
    for imgs, _ in progress_bar(test_loader):
        logits = final_model(imgs.to(DEVICE))
        probs = torch.sigmoid(logits).cpu()
        test_probs.append(probs)

test_probs = torch.cat(test_probs).numpy()
test_preds = (test_probs >= final_best_thresholds).astype(int)

# Format for submission CSV
predicted_labels_tuple = mlb.inverse_transform(test_preds)
# The first two column names of test_df are the coordinates
coord_cols = test_df.columns[:2].tolist()
output_test_df = test_df[coord_cols].copy()
output_test_df['labels'] = [';'.join(labels) for labels in predicted_labels_tuple]

test_labels_output_path = RESULTS_DIR / 'test_labels.csv'
output_test_df.to_csv(test_labels_output_path, index=False)
print(f"Saved test predictions to: {test_labels_output_path}")
display(output_test_df.head())


# 2. Generate `generated_labels.csv`
print("\nGenerating predictions for generated_labels.csv...")
# We already have the unlabeled_loader from the pseudo-labeling phase
# If the kernel was restarted, recreate it:
if 'unlabeled_loader' not in locals():
    unlabeled_df = images_df[images_df['subset'] == 'unl'].copy().reset_index(drop=True)
    unlabeled_df['multi_hot'] = [[0] * len(LABEL_CLASSES)] * len(unlabeled_df)
    unlabeled_dataset = RadioDataset(unlabeled_df, transform=val_transforms)
    unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=BATCH_SIZE*2, shuffle=False, num_workers=2, pin_memory=True)

unlabeled_probs_final = []
with torch.no_grad():
    for imgs, _ in progress_bar(unlabeled_loader):
        logits = final_model(imgs.to(DEVICE))
        probs = torch.sigmoid(logits).cpu()
        unlabeled_probs_final.append(probs)

unlabeled_probs_final = torch.cat(unlabeled_probs_final).numpy()
unlabeled_preds_final = (unlabeled_probs_final >= final_best_thresholds).astype(int)

# Format for submission CSV
unlabeled_labels_tuple = mlb.inverse_transform(unlabeled_preds_final)
unlabeled_df = images_df[images_df['subset'] == 'unl'].copy().reset_index(drop=True)

output_generated_df = pd.DataFrame()
output_generated_df[['ra', 'dec']] = pd.DataFrame(unlabeled_df['coords'].tolist(), index=unlabeled_df.index)
output_generated_df['labels'] = [';'.join(labels) for labels in unlabeled_labels_tuple]

generated_labels_output_path = RESULTS_DIR / 'generated_labels.csv'
output_generated_df.to_csv(generated_labels_output_path, index=False)
print(f"Saved generated labels for all unlabeled sources to: {generated_labels_output_path}")
display(output_generated_df.head())

print("\nAll coding deliverables are complete!")

## Visualizations

In [ ]:
# Visualization Setup

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams['figure.dpi'] = 150 # Produces high-resolution images
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'DejaVu Sans' # A safe fallback font in Colab
# import warnings
# warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# Data from Our Final Results

# 1. Class Distribution Data (from Cell 8 output)
class_names = ['Bent', 'Exotic', 'Fr I', 'Fr Ii', 'Point Source', 'S Z Shaped', 'Should Be Discarded', 'Typical', 'X Shaped']
# These are the exact counts from your training set before pseudo-labeling.
class_counts = train_df['multi_hot'].sum(axis=0)

# 2. Overall Performance Comparison Data
baseline_macro_f1 = 0.4179
final_macro_f1 = 0.5326
model_names = ['Baseline Model', 'Final Enhanced Model']
scores = [baseline_macro_f1, final_macro_f1]

# 3. Per-Class F1 Score Improvement Data
baseline_per_class_f1 = [0.5224, 0.5000, 0.6456, 0.6977, 0.7059, 0.1818, 0.1000, 0.3077, 0.0000]
final_per_class_f1 = [0.5909, 1.0000, 0.6897, 0.7363, 0.7714, 0.3333, 0.1000, 0.5714, 0.0000]

# 4. Optimal Thresholds Data from the Final Model
final_thresholds = [0.45, 0.78, 0.49, 0.65, 0.62, 0.83, 0.05, 0.86, 0.05]

# Visualization 1: The Core Challenge - Class Imbalance
plt.figure(figsize=(12, 6))
ax1 = sns.barplot(x=class_names, y=class_counts, palette="plasma")

ax1.set_title('The Core Challenge: Severe Class Imbalance in the Labeled Dataset',
              fontsize=16, fontweight='bold', pad=20)
ax1.set_xlabel('Radio Source Class', fontsize=12)
ax1.set_ylabel('Number of Images (Log Scale)', fontsize=12)
ax1.set_yscale('log')
ax1.margins(y=0.1)

plt.xticks(rotation=45, ha='right', fontsize=10)

for p in ax1.patches:
    ax1.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', fontsize=10, color='black', xytext=(0, 8),
                 textcoords='offset points')

plt.tight_layout(pad=1.5)
plt.savefig(f"{RESULTS_DIR}/1_class_distribution.png")
print(f"Saved '{RESULTS_DIR}/1_class_distribution.png'")
plt.show()


# Visualization 2: The Story of Success - Overall Performance Leap
plt.figure(figsize=(8, 6))
ax2 = sns.barplot(x=model_names, y=scores, palette="cividis", width=0.5)
ax2.set_title('Impact of Pseudo-Labeling: A Leap in Model Performance', fontsize=16, fontweight='bold', pad=20)
ax2.set_ylabel('Optimized Macro F1-Score', fontsize=12)
ax2.set_ylim(0, 0.65)
for i, score in enumerate(scores):
    improvement_text = ""
    if i > 0:
        improvement = ((score - scores[0]) / scores[0]) * 100
        improvement_text = f'\n(+{improvement:.1f}%)'
    ax2.text(i, score + 0.015, f'{score:.4f}{improvement_text}', ha='center', fontsize=12, fontweight='bold')
plt.xticks(fontsize=12)
plt.tight_layout(pad=1.5)
plt.savefig(f"{RESULTS_DIR}/2_overall_performance.png")
print(f"Saved '{RESULTS_DIR}/2_overall_performance.png'")
plt.show()


# Visualization 3: The Deep Dive - Per-Class F1 Score Improvement
per_class_df = pd.DataFrame({
    'Class': class_names,
    'Baseline Model': baseline_per_class_f1,
    'Final Enhanced Model': final_per_class_f1
})
per_class_df_melted = per_class_df.melt(id_vars='Class', var_name='Model', value_name='F1-Score')

plt.figure(figsize=(14, 7))
ax3 = sns.barplot(x='Class', y='F1-Score', hue='Model', data=per_class_df_melted, palette="viridis")
ax3.set_title('Deep Dive: How Pseudo-Labeling Boosted Performance Across Classes', fontsize=16, fontweight='bold', pad=20)
ax3.set_xlabel('Radio Source Class', fontsize=12)
ax3.set_ylabel('Per-Class F1-Score', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.legend(title='Model Type', fontsize=10)
plt.ylim(0, 1.1)
for p in ax3.patches:
    if p.get_height() > 0.01:
        ax3.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='center', fontsize=9, color='black', xytext=(0, 5),
                     textcoords='offset points')
plt.tight_layout(pad=1.5)
plt.savefig(f"{RESULTS_DIR}/3_per_class_improvement.png")
print(f"Saved '{RESULTS_DIR}/3_per_class_improvement.png'")
plt.show()


# Visualization 4: The Key Insight - Optimized Prediction Thresholds
threshold_df = pd.DataFrame({'Class': class_names, 'Threshold': final_thresholds}).sort_values('Threshold', ascending=False)

plt.figure(figsize=(12, 6))
ax4 = sns.barplot(x='Class', y='Threshold', data=threshold_df, palette="magma")
ax4.axhline(0.5, ls='--', color='red', lw=1.5, label='Default Threshold (0.5)')
ax4.set_title('A Key Insight: Optimal Prediction Thresholds Are Not One-Size-Fits-All', fontsize=16, fontweight='bold', pad=20)
ax4.set_xlabel('Radio Source Class', fontsize=12)
ax4.set_ylabel('Optimized Threshold', fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.legend()
for p in ax4.patches:
    ax4.annotate(f'{p.get_height():.2f}', (p.get_x() + p.get_width() / 2., p.get_height()),
                 ha='center', va='center', fontsize=10, color='white', xytext=(0, -12),
                 textcoords='offset points', fontweight='bold')
plt.tight_layout(pad=1.5)
plt.savefig(f"{RESULTS_DIR}/4_optimal_thresholds.png")
print(f"Saved '{RESULTS_DIR}/4_optimal_thresholds.png'")
plt.show()

## Automated Hyper-parameter searching using Optuna
- if you want to run this phase to see how we selected our hyperparametes ensure the Optuna dependency is installed and imported on in cell 4 right before the "Project Setup & Imports" step, the uncomment the 3 next cells

In [ ]:
# def objective(trial):
#     """
#     This function defines a single training trial for Optuna.
#     """
#     # 1. Define the search space for our hyperparameters
#     finetune_lr = trial.suggest_float("finetune_lr", 1e-5, 1e-3, log=True)
#     weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
#     mixup_alpha = trial.suggest_float("mixup_alpha", 0.1, 0.4)
#     smoothing_factor = trial.suggest_float("smoothing_factor", 0.05, 0.2)

#     # 2. Create the model for this trial
#     model = get_efficientnetv2(len(LABEL_CLASSES))

#     # 3. Run the training using the suggested hyperparameters
#     # NOTE: We're running a shorter trial to speed up the search process.
#     model, history = train_model(
#         model=model,
#         train_loader=train_loader,
#         val_loader=val_loader,
#         epochs=15,              # Shorter run for each trial
#         head_epochs=5,
#         head_lr=1e-3,           # Keep head_lr fixed, we are tuning the fine-tune phase
#         finetune_max_lr=finetune_lr,
#         weight_decay=weight_decay,
#         patience=8,
#         model_save_path=f'{RESULTS_DIR}/trial_{trial.number}_best_model.pth', # Save each trial's model
#         use_label_smoothing=True,
#         smoothing_factor=smoothing_factor,
#         use_mixup=True,
#         mixup_alpha=mixup_alpha,
#         accumulation_steps=4
#     )

#     # 4. Return the metric we want to maximize
#     best_f1_in_trial = max(history['val_macro_f1']) if history['val_macro_f1'] else 0

#     # Optional: Pruning (stops unpromising trials early)
#     trial.report(best_f1_in_trial, step=len(history['val_macro_f1']))
#     if trial.should_prune():
#         raise optuna.exceptions.TrialPruned()

#     return best_f1_in_trial

In [ ]:
# # 1. Define the storage location in your Google Drive
# storage_path = f"sqlite:///{RESULTS_DIR}/cos711_study.db"
# study_name = "efficientnetv2-m-tuning" # You can name your study

# print(f"Optuna study results will be saved to: {storage_path}")

# # 2. Create the study with persistent storage
# # load_if_exists=True is key: it allows you to resume if the session disconnects.
# study = optuna.create_study(
#     study_name=study_name,
#     storage=storage_path,
#     direction="maximize",
#     pruner=optuna.pruners.MedianPruner(),
#     load_if_exists=True
# )

# # 3. Run the optimization
# # This will automatically save progress after each trial and can be resumed.
# study.optimize(objective, n_trials=20)

# # 4. Save the final results and print the best trial
# print("\n Optimization Complete ")

# # Save a human-readable CSV of all trial results
# results_df = study.trials_dataframe()
# results_df_path = RESULTS_DIR / 'optuna_results.csv'
# results_df.to_csv(results_df_path, index=False)
# print(f"Full study results saved to: {results_df_path}")

# # Print the best results to the console
# print(f"\nBest trial number: {study.best_trial.number}")
# print(f"Best validation Macro F1: {study.best_value:.4f}")
# print("Best hyperparameters found:")
# for key, value in study.best_params.items():
#     print(f"  - {key}: {value}")